In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import einops as eo
import os
import rasterio as rio
from rasterio.features import rasterize
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

from c2s.models.chicken import ChickenConfig, ChickenOutput, Chicken

In [2]:
# Data pipeline parameters

# Data directory - should contain a subdirectory for each site
data_dir = '/network/scratch/m/matthew.fortier/lichen'

# File to put CSV data into
csv_file = 'dataset_chm_shp.csv'

# Classes can be commented out here if we wish to exclude them
class_map = {
    'lichen': 1,
    'rock': 4,
    'broadleaf': 5,
    'needleleaf': 6,
    'deadwood': 7,
    'graminoids': 8,
    'moss': 9,
    'soil': 10,
    'low_green': 12,
    'dry_branches': 13,
}

# which size chicken to use
vit_size = "large"

In [3]:
sites = os.listdir(data_dir)
print(sites)
#sites = ['CS3A'] # override with a single site for testing purposes

['CG1-8A', 'CG1-8B', 'CS-103A', 'CS-103B', 'CS117B', 'CS3A', 'CS3B', 'CS3C', 'CS-46A', 'CS-46B', 'CS-59B', 'CS-96B', 'F3-19B', 'F3-19C', 'F3-20A', 'F3-20B', 'F3-20C', 'F3-8A', 'F3-8B', 'F3-8C', 'ZF20-11A', 'ZF46-15A', 'ZF46-37A', 'ZF46-45A']


In [6]:
def load_site_data(site: str):
    """Retrieves all the rgb / canopy height / label data for a drone site"""
    
    rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
    chm_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.tif')
    chm_shp_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.shp')
    label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

    # load RGB raster and metadata
    with rio.open(rgb_file) as f:
        # Get raster metadata
        height = f.height
        width = f.width
        crs = f.crs
        transform = f.transform
        rgb_raster = f.read((1,2,3))
    
    '''
    # load chm file
    with rio.open(chm_file) as f:
        chm_raster = f.read(
            1,
            out_shape=(
                height,
                width
            ),
            resampling=Resampling.nearest
        ).astype(int)
    '''
        
    # load chm as shapefile and rasterize
    chm_gdf = gpd.read_file(chm_shp_file).to_crs(crs)
    chm_raster = rasterize(
        ((r['geometry'], r['DN']) for _, r in chm_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform
    )
    
    # labels are stored as shapefiles, so we will need to rasterize it
    label_gdf = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class'])
    label_raster = rasterize(
        ((r['geometry'], r['class']) for _, r in label_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform,
        dtype='uint8'
    )
    certainty_raster = rasterize(
        ((r['geometry'], r['certainty']) for _, r in label_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform,
        dtype='uint8'
    )
    
    return rgb_raster, chm_raster, label_raster, certainty_raster

In [10]:
def extract_pixel_features(rgb, chm, labels, certainty):
    # Get a list of indices where we have labelled pixels
    # Ex: [ [348, 1451], [581, 2099], ... ]
    indices = np.argwhere(np.isin(labels, list(class_map.values())))
    
    rgb_features = rgb[:,indices[:,0],indices[:,1]].T
    rgb_df = pd.DataFrame(rgb_features, columns=['R', 'G', 'B'])
    
    chm_features = chm[indices[:,0],indices[:,1]].reshape(-1, 1)
    chm_df = pd.DataFrame(chm_features, columns=['chm'])
    
    labels_array = labels[indices[:,0],indices[:,1]].reshape(-1, 1)
    labels_df = pd.DataFrame(labels_array, columns=['veg_class'])
    
    certainty_array = certainty[indices[:,0],indices[:,1]].reshape(-1, 1)
    certainty_df = pd.DataFrame(labels_array, columns=['veg_class'])
    
    df = pd.concat([labels_df, certainty_df, rgb_df, chm_df], axis=1)
    return df, indices

In [11]:
def get_chicken_feature_map(rgb_raster, vit_size="small"):
    """ Use sliding-window DinoV2 to extract visual features
    rgb_raster: torch tensor with shape (B, C, H, W)
    returns:    torch tensor with shape (B, num_features, H, W)
    """
    rgb_torch_raster = torch.from_numpy(rgb_raster.astype(np.float32)).unsqueeze(0)
    b, c, h, w = rgb_torch_raster.shape
    new_h = int(h/14)*14
    new_w = int(w/14)*14
    rgb_cropped = rgb_torch_raster[:,:,:new_h,:new_w]
    chicken_config = ChickenConfig(
        vit_size=vit_size,
        slider_buffer_size=1
    )
    chicken = Chicken(chicken_config)
    op = chicken._chicken_slider_infer(rgb_cropped).detach().numpy()
    return op

In [12]:
def extract_chicken_pixel_features(chicken_feature_raster, indices):
    # Find the relevant feature vector for labeled pixels. Since texture featuremap is downsampled,
    # we need to modify the indices we use.
    chicken_indices = (indices/14).astype(int)
    chicken_pixel_features = chicken_feature_raster[0][:,chicken_indices[:,0],chicken_indices[:,1]].T
    chicken_df = pd.DataFrame(chicken_pixel_features, columns=[f'chicken_{i}' for i in range(chicken_pixel_features.shape[1])])
    return chicken_df

In [13]:
all_features = None

In [15]:
for site in sites:
    print(f'Processing {site}...')
    rgb, chm, labels, certainties = load_site_data(site)
    df1, indices = extract_pixel_features(rgb, chm, labels, certainties)
    df1['site'] = site
    
    # texture features
    chicken_feature_map = get_chicken_feature_map(rgb, vit_size=vit_size)
    df2 = extract_chicken_pixel_features(chicken_feature_map, indices)

    df = pd.concat([df1,df2], axis=1)

    if all_features is None:
        all_features = df
    else:
        all_features = pd.concat([all_features, df], axis=0)
    

Processing CG1-8A...


Using cache found in /home/mila/m/matthew.fortier/.cache/torch/hub/facebookresearch_dinov2_main
xFormers not available
xFormers not available


Processing CG1-8B...


Using cache found in /home/mila/m/matthew.fortier/.cache/torch/hub/facebookresearch_dinov2_main


Processing CS-103A...


...

Processing ZF46-15A...


Using cache found in /home/mila/m/matthew.fortier/.cache/torch/hub/facebookresearch_dinov2_main


Processing ZF46-37A...


/home/mila/m/matthew.fortier/.conda/envs/scratch/lib/python3.9/site-packages/rasterio/dtypes.py:205: RuntimeWarning: invalid value encountered in cast
  return np.allclose(values, values.astype(dtype), equal_nan=True)


ValueError: shape values cannot be cast to specified dtype: uint8

In [24]:
all_features['site'].value_counts()

site
CS3A        1784
ZF46-15A    1308
ZF46-37A    1194
CS-59B      1193
ZF46-45A    1188
F3-19C      1175
F3-20A      1167
CS-103B     1165
F3-8B       1160
F3-20C      1159
CS3C        1155
ZF20-11A    1155
F3-8C       1154
F3-20B      1152
F3-19B      1146
CS-46A      1137
F3-8A       1129
CS117B      1128
CS-96B      1113
CG1-8A      1110
CS-103A     1109
CS3B        1070
CG1-8B      1041
CS-46B      1038
Name: count, dtype: int64

In [26]:
# Some additional cleanup
df = all_features
# Remove -9999 values from chm
df['chm'][df['chm'] == 0] = -1

df['chm'].value_counts()

/tmp/ipykernel_3079818/672315034.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['chm'][df['chm'] == 0] = -1


chm
 1    18320
 2     5383
 3     4308
-1      119
Name: count, dtype: int64

In [27]:
# Normalize colors
for c in ['R', 'G', 'B']:
    df[c] = df[c].astype(float)/255.0

In [31]:
df

,R,G,B,chm,veg_class,site,chicken_0,chicken_1,chicken_2,chicken_3,...,chicken_1014,chicken_1015,chicken_1016,chicken_1017,chicken_1018,chicken_1019,chicken_1020,chicken_1021,chicken_1022,chicken_1023
0,0.109804,0.239216,0.129412,1,5,CS-103A,-0.263335,-0.924741,0.453007,0.793199,...,1.055867,-0.581324,-0.221842,0.435689,-0.043067,0.593084,-1.062014,0.788338,-0.218863,0.008741
1,0.082353,0.211765,0.105882,1,5,CS-103A,-0.263335,-0.924741,0.453007,0.793199,...,1.055867,-0.581324,-0.221842,0.435689,-0.043067,0.593084,-1.062014,0.788338,-0.218863,0.008741
2,0.098039,0.223529,0.113725,1,5,CS-103A,-0.263335,-0.924741,0.453007,0.793199,...,1.055867,-0.581324,-0.221842,0.435689,-0.043067,0.593084,-1.062014,0.788338,-0.218863,0.008741
3,0.082353,0.211765,0.101961,1,5,CS-103A,-0.263335,-0.924741,0.453007,0.793199,...,1.055867,-0.581324,-0.221842,0.435689,-0.043067,0.593084,-1.062014,0.788338,-0.218863,0.008741
4,0.388235,0.505882,0.364706,3,6,CS-103A,0.121613,-1.450172,-0.453014,0.432836,...,2.425043,-1.036681,-0.759858,0.862493,0.936467,0.815851,2.938023,-0.005285,3.091863,0.218643
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1036,0.788235,0.870588,0.780392,3,6,CG1-8B,0.282377,-2.179713,-0.477716,1.279851,...,1.875949,-0.792436,0.067023,0.707657,1.154750,-0.361834,0.993847,0.094023,1.211099,-0.067749
1037,0.792157,0.858824,0.752941,2,6,CG1-8B,-0.114267,-2.105973,0.030556,-0.633501,...,-0.021213,0.048232,-0.330421,-0.257352,-0.850836,-0.990270,0.750098,0.763686,1.430246,-0.166895
1038,0.741176,0.807843,0.678431,2,6,CG1-8B,-0.114267,-2.105973,0.030556,-0.633501,...,-0.021213,0.048232,-0.330421,-0.257352,-0.850836,-0.990270,0.750098,0.763686,1.430246,-0.166895
1039,0.674510,0.749020,0.639216,2,6,CG1-8B,-0.114267,-2.105973,0.030556,-0.633501,...,-0.021213,0.048232,-0.330421,-0.257352,-0.850836,-0.990270,0.750098,0.763686,1.430246,-0.166895


In [32]:
df.to_csv(csv_file, index=False)

<bound method NDFrame.head of              R         G         B  chm  veg_class     site  chicken_0   
0     0.109804  0.239216  0.129412    1          5  CS-103A  -0.263335  \
1     0.082353  0.211765  0.105882    1          5  CS-103A  -0.263335   
2     0.098039  0.223529  0.113725    1          5  CS-103A  -0.263335   
3     0.082353  0.211765  0.101961    1          5  CS-103A  -0.263335   
4     0.388235  0.505882  0.364706    3          6  CS-103A   0.121613   
...        ...       ...       ...  ...        ...      ...        ...   
1036  0.788235  0.870588  0.780392    3          6   CG1-8B   0.282377   
1037  0.792157  0.858824  0.752941    2          6   CG1-8B  -0.114267   
1038  0.741176  0.807843  0.678431    2          6   CG1-8B  -0.114267   
1039  0.674510  0.749020  0.639216    2          6   CG1-8B  -0.114267   
1040  0.788235  0.858824  0.733333    2          6   CG1-8B  -0.114267   

      chicken_1  chicken_2  chicken_3  ...  chicken_1014  chicken_1015   
0     -